In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import hashlib
import zlib
import base64

In [2]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

In [3]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

In [4]:
fecha_ini = datetime.strptime('2023-01-01', '%Y-%m-%d')
fecha_fin = datetime.strptime('2024-12-31', '%Y-%m-%d')
delta = timedelta(days=15)

current_start_date = fecha_ini

while current_start_date < fecha_fin:
    print(f'Procesando desde {current_start_date}')
    current_end_date = min(current_start_date + delta, fecha_fin)

    query = f"""
    SELECT 
    R.REDASISCOD, 
    R.REDASISDES, 
    C.CENASICOD, 
    C.CENASIDES, 
    A.ATENOMACTMEDNUM, 
    A.ATENOMFEC, 
    A.ATENOMHOR, 
    CS.CSECOD, 
    CS.CSEDES, 
    CP.CPSCOD, 
    CP.CPSDES,
    RA.RESATENOMEDCOD,
    RA.RESATENOMEDDES,
    AH.AREHOSCOD,
    AH.AREHOSDES,
    SH.SERVHOSCOD,
    SH.SERVHOSDES,
    ACT.ACTCOD,
    ACT.ACTDES,
    ACTE.ACTESPCOD,
    ACTE.ACTESPNOM,
    TD.TIPDOCIDENPERNOM,
    A.ATENOMPROPERASISDOCIDENNUM,
    PER.PERTIPDOCIDENCOD,
    PER.PERDOCIDENNUM,
    PER.PERNACFEC,
    PER.PERSEXOCOD,
    DIA.DIAGCOD,
    DIA.DIAGDES,
    D.ATENMDDIAGORD,
    TDIA.TIPODIAGCOD,
    TDIA.TIPODIAGNOM,
    CDIA.CASODIAGCOD,
    CDIA.CASODIAGNOM
    FROM CTANM10 A
            LEFT OUTER JOIN
            CTDAN10 D
                ON A.ATENOMORICENASICOD = D.ATENOMORICENASICOD 
                AND A.ATENOMCENASICOD = D.ATENOMCENASICOD 
                AND A.ATENOMACTMEDNUM = D.ATENOMACTMEDNUM
            LEFT OUTER JOIN
            CMCAS10 C
                ON A.ATENOMCENASICOD = C.CENASICOD
            LEFT OUTER JOIN
            CMRAS10 R
                ON C.REDASISCOD = R.REDASISCOD
            LEFT OUTER JOIN 
            CMCSP10 CS
                ON A.ATENOMCSECOD = CS.CSECOD
            LEFT OUTER JOIN 
            CMCPP10 CP
                ON A.ATENOMCPSCOD = CP.CPSCOD
            LEFT OUTER JOIN 
            CBRAN10 RA
                ON A.RESATENOMEDCOD = RA.RESATENOMEDCOD
            LEFT OUTER JOIN
            CMAHO10 AH
                ON A.ATENOMPROAREHOSCOD = AH.AREHOSCOD
            LEFT OUTER JOIN
            CMSHO10 SH
                ON A.ATENOMPROSERVHOSCOD = SH.SERVHOSCOD 
            LEFT OUTER JOIN
            CMACT10 ACT
                ON A.ATENOMPROACTCOD = ACT.ACTCOD  
            LEFT OUTER JOIN
            CMACE10 ACTE
                ON A.ATENOMPROACTCOD = ACTE.ACTCOD
                AND A.ATENOMPROACTESPCOD = ACTE.ACTESPCOD
            LEFT OUTER JOIN
            CBTDI10 TD
                ON A.ATENOMPROTIPDOCIDENPERCOD = TD.TIPDOCIDENPERCOD
            LEFT OUTER JOIN
            CMDIA10 DIA
                ON D.ATENMDDIAGCOD = DIA.DIAGCOD
            LEFT OUTER JOIN
            CBTID10 TDIA
                ON D.ATENMDTIPODIAGCOD = TDIA.TIPODIAGCOD
            LEFT OUTER JOIN
            CBCAD10 CDIA
                ON D.ATENMDCASODIAGCOD = CDIA.CASODIAGCOD
            LEFT OUTER JOIN
            CMPER10 PER
                ON A.ATENOMPACSECNUM = PER.PERSECNUM
    WHERE 
        (A.ATENOMFEC >= TO_DATE('{current_start_date.strftime('%Y-%m-%d')}', 'YYYY-MM-DD') 
        AND A.ATENOMFEC < TO_DATE('{current_end_date.strftime('%Y-%m-%d')}', 'YYYY-MM-DD'))
        AND (
            A.ATENOMCPSCOD  IN ('99402.14' , '90860')
            OR A.ATENOMPROACTCOD IN ('91', 'B1')
            OR A.ATENOMPROSERVHOSCOD IN ('AB1', 'AM3', 'E21', 'F11', 'F21', 'AH1')
            OR D.ATENMDDIAGCOD IN ('%I10%', '%E10%', '%E11%', '%E13%', '%E14%', '%N18%', 'Z13.3', 'U14.0', '%O80%', '%O81%', '%O82%', '%O83%', '%O84%', '%Z39%','%F32%', 'F53.0', 'F23.1', 'F32.2', '%F31%')
        ) 
    """
    # Ejecutar la consulta y guardar los resultados
    base = pd.read_sql_query(query, con=connection0)

    # Eliminar columnas duplicadas
    base = base.loc[:,~base.columns.duplicated()]

    print(base.shape)

    base = base[base['perdocidennum'].apply(lambda x: str(x).isdigit())]
    
    print(base.shape)

    base['nuevo'] = base['pertipdocidencod'] + base['perdocidennum']

    # Convertir la columna original a bytes
    base['id_paciente'] = (base['nuevo'].astype(int)*base['nuevo'].astype(int)).astype(str).apply(lambda x: x.encode())
    # Comprimir la columna original
    base['id_paciente'] = base['id_paciente'].apply(lambda x: zlib.compress(x))
    # Convertir la columna comprimida a base64 y luego a un string
    base['id_paciente'] = base['id_paciente'].apply(lambda x: base64.b64encode(x).decode())

    base.drop('nuevo', axis=1)
    base.drop('pertipdocidencod', axis=1)
    base.drop('perdocidennum', axis=1)

    # Subir los datos a la base de datos 'ietsi_depresion'
    base.to_sql(name='ietsi_depresion', con=connection1, if_exists='append', index=False, chunksize=10000)
    print(f'Subido correctamente {current_start_date}')

    # Mover al siguiente bloque de 10 días
    current_start_date = current_end_date


Procesando desde 2023-01-01 00:00:00
(756468, 34)
(756040, 34)
Subido correctamente 2023-01-01 00:00:00
Procesando desde 2023-01-16 00:00:00
(885230, 34)
(884811, 34)
Subido correctamente 2023-01-16 00:00:00
Procesando desde 2023-01-31 00:00:00
(937345, 34)
(936921, 34)
Subido correctamente 2023-01-31 00:00:00
Procesando desde 2023-02-15 00:00:00
(942304, 34)
(941955, 34)
Subido correctamente 2023-02-15 00:00:00
Procesando desde 2023-03-02 00:00:00
(907082, 34)
(906765, 34)
Subido correctamente 2023-03-02 00:00:00
Procesando desde 2023-03-17 00:00:00
(911394, 34)
(911054, 34)
Subido correctamente 2023-03-17 00:00:00
Procesando desde 2023-04-01 00:00:00
(850147, 34)
(849888, 34)
Subido correctamente 2023-04-01 00:00:00
Procesando desde 2023-04-16 00:00:00
(956140, 34)
(955866, 34)
Subido correctamente 2023-04-16 00:00:00
Procesando desde 2023-05-01 00:00:00
(864732, 34)
(864473, 34)
Subido correctamente 2023-05-01 00:00:00
Procesando desde 2023-05-16 00:00:00
(973469, 34)
(973194, 34)
S

KeyError: 'pertipdocidencod'

In [ ]:
base['nuevo'] = base['pertipdocidencod'] + base['perdocidennum']

# Convertir la columna original a bytes
base['id_paciente'] = (base['nuevo'].astype(int)*base['nuevo'].astype(int)).astype(str).apply(lambda x: x.encode())
# Comprimir la columna original
base['id_paciente'] = base['id_paciente'].apply(lambda x: zlib.compress(x))
# Convertir la columna comprimida a base64 y luego a un string
base['id_paciente'] = base['id_paciente'].apply(lambda x: base64.b64encode(x).decode())

base = base.drop('nuevo', axis=1)
base = base.drop('pertipdocidencod', axis=1)
base = base.drop('perdocidennum', axis=1)

,redasiscod,redasisdes,cenasicod,cenasides,atenomactmednum,atenomfec,atenomhor,csecod,csedes,cpscod,...,persexocod,diagcod,diagdes,atenmddiagord,tipodiagcod,tipodiagnom,casodiagcod,casodiagnom,nuevo,id_paciente
0,18,RED ASISTENCIAL AREQUIPA,003,H.N. CARLOS ALBERTO SEGUIN ESCOBEDO,4860066,2023-01-01,0001-01-01 16:42:02,A4,(2008) CARTERA DE SERVICIOS DEL ADULTO DE 50 A...,E0100,...,0,N18.6,ENFERMEDAD RENAL CRONICA ESTADIO 5 EN DIALISIS,1.0,2,DEFINITIVO,2,REPETIDO,129233300,eJwzNDM3MDQyMbUwsrCwNAACAB8LA24=
1,18,RED ASISTENCIAL AREQUIPA,003,H.N. CARLOS ALBERTO SEGUIN ESCOBEDO,4860066,2023-01-01,0001-01-01 16:42:02,A4,(2008) CARTERA DE SERVICIOS DEL ADULTO DE 50 A...,E0100,...,0,Z49.0,CUIDADOS PREPARATORIOS PARA DIALISIS,2.0,2,DEFINITIVO,2,REPETIDO,129233300,eJwzNDM3MDQyMbUwsrCwNAACAB8LA24=
2,18,RED ASISTENCIAL AREQUIPA,003,H.N. CARLOS ALBERTO SEGUIN ESCOBEDO,4860066,2023-01-01,0001-01-01 16:42:02,A4,(2008) CARTERA DE SERVICIOS DEL ADULTO DE 50 A...,E0100,...,0,Z49.1,DIALISIS EXTRACORPOREA,3.0,2,DEFINITIVO,2,REPETIDO,129233300,eJwzNDM3MDQyMbUwsrCwNAACAB8LA24=
3,18,RED ASISTENCIAL AREQUIPA,003,H.N. CARLOS ALBERTO SEGUIN ESCOBEDO,4860066,2023-01-01,0001-01-01 16:42:02,A4,(2008) CARTERA DE SERVICIOS DEL ADULTO DE 50 A...,E0100,...,0,Z99.2,DEPENDENCIA DE DIALISIS RENAL,4.0,2,DEFINITIVO,2,REPETIDO,129233300,eJwzNDM3MDQyMbUwsrCwNAACAB8LA24=
4,18,RED ASISTENCIAL AREQUIPA,003,H.N. CARLOS ALBERTO SEGUIN ESCOBEDO,4860026,2023-01-01,0001-01-01 12:25:29,A6,(2008) CARTERA DE SERVICIOS DEL ADULTO DE 65 A...,E0100,...,1,N18.6,ENFERMEDAD RENAL CRONICA ESTADIO 5 EN DIALISIS,1.0,2,DEFINITIVO,2,REPETIDO,129677509,eJwzNLMwNDMyNTM2MTAxMTWwMAQAHx0DcQ==
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,28,RED ASISTENCIAL PASCO,341,CAP II YANAHUANCA,156698,2023-01-01,0001-01-01 15:23:46,A4,(2008) CARTERA DE SERVICIOS DEL ADULTO DE 50 A...,99401,...,0,M62.4,CONTRACTURA MUSCULAR,1.0,1,PRESUNTIVO,1,NUEVO,104211592,eJwzNLAwMzAwNbU0MDc0NzExMwEAHt4DdA==
356,28,RED ASISTENCIAL PASCO,341,CAP II YANAHUANCA,156699,2023-01-01,0001-01-01 15:24:42,B4,(2014) CARTERA DE SERVICIOS EN ADOLESCENTE DE ...,99401,...,0,M62.4,CONTRACTURA MUSCULAR,1.0,1,PRESUNTIVO,1,NUEVO,171784715,eJwzsjQ1sLS0sDA2MDczNjQyMgUAH9kDgA==
357,28,RED ASISTENCIAL PASCO,341,CAP II YANAHUANCA,156700,2023-01-01,0001-01-01 15:26:16,A7,(2008) CARTERA DE SERVICIOS DEL ADULTO DE 70 A...,99401,...,0,M25.5,DOLOR EN ARTICULACION,1.0,1,PRESUNTIVO,1,NUEVO,104202156,eJwzNLAwtTCwsDQ2NDUwsTA2NgMAHzkDeQ==
358,28,RED ASISTENCIAL PASCO,341,CAP II YANAHUANCA,156701,2023-01-01,0001-01-01 15:27:36,A5,(2008) CARTERA DE SERVICIOS DEL ADULTO DE 60 A...,99401,...,1,M54.9,"DORSALGIA, NO ESPECIFICADA",2.0,1,PRESUNTIVO,1,NUEVO,104200026,eJwzNLAwNTczMTUxtDAxMDAzNwMAHzMDeQ==


select 
id.redasiscod as COD_RED,
id.redasisdes as RED,
id.cenasicod as COD_CAS,
id.cenasides as IPRESS,
id.atenomactmednum as ACTO_MEDICO,
id.atenomfec as FECHA_ATENCION,
id.atenomhor as HORA_ATENCION,
id.csecod as COD_CARTERA_SERVICIOS,
id.cpsdes as CARTERA_SREVICIOS,
id.cpscod as COD_PROCEDIMIENTO,
id.cpsdes as PROCEDIMIENTO,
id.resatenomedcod as COD_RESULTADO_ATENCION,
id.resatenomeddes as RESULTADO_ATENCION,
id.arehoscod as COD_AREA,
id.arehosdes as AREA_HOSPITALARIA,
id.servhoscod as COD_SERVICIO,
id.servhosdes as SERVICIO,
id.actcod as COD_ACTIVIDAD,
id.actdes as ACTIVIDAD,
id.actespcod as COD_ACTIVIDAD_ESPECIFICA,
id.actespnom as ACTIVIDAD_ESPECIFICA,
id.tipdocidenpernom as TIPO_DOCUMENTO_PRS_ASISTENCIAL,
id.atenomproperasisdocidennum as NUMERO_DOCUMENTO_PRS_ASISTENCIAL,
id.id_paciente as ID_PACIENTE,
id.pernacfec as FEC_NAC_PACIENTE,
CASE 
        WHEN id.persexocod = '0' THEN 'MASCULINO'
        WHEN id.persexocod = '1' THEN 'FEMENINO'
        ELSE 'OTRO'
    END AS SEXO_PACIENTE,
id.diagcod as COD_DIAGNOSTICO,
id.diagdes as DIAGNOSTICO,
id.atenmddiagord as ORDEN_DIAGNOSTICO,
id.tipodiagcod as COD_TIPO_DIAGNOSTICO,
id.tipodiagnom as TIPO_DIAGNOSTICO,
id.casodiagcod as COD_CASO_DIAGNOSTICO,
id.casodiagnom as CASO_DIAGNOSTICO
from ietsi_depresion id where id.atenomfec >= '2023-01-01' and id.atenomfec < '2023-02-01'


In [ ]:
INSERT INTO homeenlaces_historico (index_homenlaces,cod_gpo_ocup,desc_gpo_ocup,cod_area,desc_area,cod_actividad,desc_actividad,cod_subactividad,desc_subactividad,cod_servicio,desc_servicio,cod_especialidad,especialidad,cod_subespecialidad,subespecialidad,cod_agrupador,agrupador,cod_variable,variable,anio) VALUES (684,'01','MEDICO','01','CONSULTA EXTERNA','91','ATENCION  MEDICA AMBULATORIA','333','TELECONSULTA','AM6','MEDICINA OCUPACIONAL Y DEL MEDIO AMBIENTE','031','MEDICINA OCUPACIONAL Y DEL MEDIO AMBIENTE','040','MEDICINA OCUPACIONAL Y DEL MEDIO AMBIENTE','006','Salud Ocupacional','023','CONSULTA MEDICA_TELECONSULTA_SALUD OCUPACIONAL','2024')

In [1]:
684,'01','MEDICO','01','CONSULTA EXTERNA','91','ATENCION  MEDICA AMBULATORIA','333','TELECONSULTA','AM6','MEDICINA OCUPACIONAL Y DEL MEDIO AMBIENTE','031','MEDICINA OCUPACIONAL Y DEL MEDIO AMBIENTE','040','MEDICINA OCUPACIONAL Y DEL MEDIO AMBIENTE','006','Salud Ocupacional','023','CONSULTA MEDICA_TELECONSULTA_SALUD OCUPACIONAL','2024',

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (336812353.py, line 2)

In [ ]:
select-- t.CITAMBPROCONCENASICOD,
t.CITAMBCENASICOD as cod_centro,
t.CITAMBNUM as acto_med,
t.CITAMBAREHOSCOD as cod_area,
t.CITAMBSERVHOSCOD as cod_servicio,
t.CITAMBACTCOD as cod_actividad,
t.CITAMBACTESPCOD as cod_subactividad,
to_char(t.citambproconfec, 'yyyymm') periodo,
t.ESTCITCOD as cod_estado,
n.tipopacicod as cod_paciente
from ctcam10 t
left outer join cmame10 k on t.citamboricenasicod  = k.oricenasicod
                         and t.citambcenasicod     = k.cenasicod
                         and t.citambnum           = k.actmednum
left outer join cbtpc10 n on k.actmedtipopacicod       = n.tipopacicod

where t.citamboricenasicod                  in ('1','2','3','4','5','6','7')
--and t.citambcenasicod                     = '296'
and to_char(trunc(t.citambproconfec),'yyyymmdd') >= '20240401'
and to_char(trunc(t.citambproconfec),'yyyymmdd') <= '20240531'

--and to_char(trunc(t.citambproconfec),'yyyymmdd') >= (SELECT to_char((CASE --'20230301'--
--    WHEN SYSDATE > TRUNC(SYSDATE, 'MM')+14 AND SYSDATE < last_day(sysdate) 
--    THEN TRUNC(SYSDATE, 'MM')
--    ELSE trunc(add_months(sysdate, -1), 'MM') + 15
--END),'yyyymmdd') AS resultado
--FROM dual)
--and to_char(trunc(t.citambproconfec),'yyyymmdd') <= (SELECT to_char((CASE --'20230331'
--    WHEN SYSDATE > TRUNC(SYSDATE, 'MM')+14 AND SYSDATE < last_day(sysdate) 
--    THEN TRUNC(SYSDATE, 'MM')+14
--    ELSE last_day(add_months(sysdate,-1))
--END),'yyyymmdd') AS resultado
--FROM dual)
and t.citambactcod = '91'
